# 13 - Full-corpus preprocessing with medprep

Generalises notebook 07 from the 58 expert-labelled studies to all 4,407 studies and
fixes two correctness gaps:

* **Orientation.** Slices are ordered by patient coordinate and every volume is
  flipped (never rotated) so its array axes increase along fixed patient directions.
  Scanners that write mirrored direction cosines then agree.
* **Laterality.** Right knees are mirrored to a left-knee convention, so "medial" is
  always on the same side of the array. The side comes from the DICOM `Laterality`
  tag, else from where the image centre lies along the patient x axis; within 20 mm
  of the midline it stays unresolved (not flipped, and counted).

**medprep** (my Med_Im_Prepper) does the per-slice DICOM read (`load_image`: modality
LUT, rejects padding and multiframe data) and the volume operations (`preprocess`:
per-volume 1-99% clip + minmax, aspect-preserving resize/pad). Nothing is fitted on
data, so there is no train/test leakage at this step.

Output per study: `(3 planes, 16 slices, 224, 224)` uint8 + plane-presence mask.
About 10.6 GB for 4,407 studies, under Kaggle's 20 GB output limit.

**Settings:** CPU, internet ON (installs medprep). Resumable: rerunning skips
finished studies. Set `LIMIT = 20` for a first smoke run.
**Outputs:** `cache/<study>.npz`, `preprocess_manifest_v1.csv`.

In [ ]:
%pip install -q "medprep-starter[medical] @ git+https://github.com/Z-bros/Med_Im_Prepper.git"

In [ ]:
# Shared preprocessing: this cell is identical in notebooks 13 and 15.
# Change PREPROC_VERSION whenever anything below changes; 15 refuses to run
# against a model trained on a different version.
import numpy as np
import pydicom
from medprep import preprocess, PreprocessConfig, load_image

PREPROC_VERSION = "v1-16x224-pad-p1_99-canonical-lps-leftknee"
PLANES = ("Sagittal", "Coronal", "Axial")
N_SLICES = 16
IMG = 224
LAT_MIN_OFFSET_MM = 20.0  # closer to the midline than this: do not guess the side


def _floats(value, n):
    out = np.asarray([float(x) for x in value], dtype=float)
    if out.shape != (n,):
        raise ValueError(f"expected {n} values, got {out.shape}")
    return out


def select_series(study_series, series_root):
    """One series per plane: fluid-sensitive + fat-suppressed preferred (as in 06),
    then most slices, then UID for a deterministic tie-break."""
    picks = {}
    for plane in PLANES:
        g = study_series[study_series["Anatomical_Plane"].eq(plane)]
        if g.empty:
            continue
        ranked = []
        for row in g.itertuples(index=False):
            folder = series_root / row.StudyInstanceUID / row.SeriesInstanceUID
            n = sum(1 for _ in folder.glob("*.dcm"))
            pref = int(row.Fluid_Sensitive == 1 and row.Fat_Suppression == 1)
            ranked.append((-pref, -n, row.SeriesInstanceUID, folder, n))
        best = sorted(ranked)[0]
        if best[4] >= 2:
            picks[plane] = best[3]
    return picks


def canonical_axes(iop):
    """Patient axis carried by (slice, row, column) array axes, and their signs.

    DICOM: iop[0:3] is the direction of increasing column index (array axis 2),
    iop[3:6] of increasing row index (array axis 1); slices are sorted below by
    ascending patient coordinate, so axis 0 is always positive.
    """
    row_dir, col_dir = iop[:3], iop[3:]
    normal = np.cross(row_dir, col_dir)
    axes = (int(np.argmax(np.abs(normal))), int(np.argmax(np.abs(col_dir))),
            int(np.argmax(np.abs(row_dir))))
    if len(set(axes)) != 3:
        raise ValueError(f"oblique orientation, axes collide: {axes}")
    signs = (1, int(np.sign(col_dir[axes[1]])), int(np.sign(row_dir[axes[2]])))
    return axes, signs


def to_canonical(vol, iop):
    """Flip (never rotate) so every array axis increases along +L, +P or +S."""
    axes, signs = canonical_axes(iop)
    if signs[1] < 0:
        vol = vol[:, ::-1, :]
    if signs[2] < 0:
        vol = vol[:, :, ::-1]
    return np.ascontiguousarray(vol), axes


def to_left_knee(vol, axes, side):
    """Mirror a right knee across the patient x axis (LPS +x = patient left)."""
    if side != "R":
        return vol
    return np.ascontiguousarray(np.flip(vol, axis=axes.index(0)))


def read_series(folder):
    """Ordered, LUT-applied, canonically flipped float32 volume plus geometry."""
    files = sorted(folder.glob("*.dcm"))
    headers = [pydicom.dcmread(f, stop_before_pixels=True) for f in files]
    for ds in headers:
        if str(getattr(ds, "PhotometricInterpretation", "")) != "MONOCHROME2":
            raise ValueError("non-MONOCHROME2 slice; medprep does not invert MONOCHROME1")
    iops = np.stack([_floats(ds.ImageOrientationPatient, 6) for ds in headers])
    if not np.allclose(iops, iops[0], atol=1e-3, rtol=0):
        raise ValueError("slice orientations differ within the series")
    iop = iops[0]
    axes, _ = canonical_axes(iop)
    ipps = np.stack([_floats(ds.ImagePositionPatient, 3) for ds in headers])
    order = np.argsort(ipps[:, axes[0]], kind="stable")
    # medprep.load_image applies the modality LUT and rejects padding/multiframe.
    slices = [load_image(files[i])[0] for i in order]
    if len({s.shape for s in slices}) != 1:
        raise ValueError("slice shapes differ within the series")
    vol, axes = to_canonical(np.stack(slices).astype(np.float32), iop)
    mid = headers[order[len(order) // 2]]
    ps = _floats(mid.PixelSpacing, 2)
    centre = (_floats(mid.ImagePositionPatient, 3)
              + iop[:3] * ps[1] * float(mid.Columns) / 2
              + iop[3:] * ps[0] * float(mid.Rows) / 2)
    tag = str(getattr(mid, "Laterality", "") or "").upper()
    return vol, {"axes": axes, "centre_x": float(centre[0]), "tag": tag,
                 "n_slices": len(files),
                 "geom_plane": {0: "Sagittal", 1: "Coronal", 2: "Axial"}[axes[0]]}


def study_side(metas):
    """Laterality tag if consistent, else median image-centre x; None if unsure."""
    tags = {m["tag"] for m in metas} - {""}
    if len(tags) == 1 and tags <= {"L", "R"}:
        return tags.pop(), "tag"
    xs = [m["centre_x"] for m in metas]
    if not xs:
        return None, "none"
    x = float(np.median(xs))
    if abs(x) < LAT_MIN_OFFSET_MM:
        return None, "unresolved"
    return ("R" if x < 0 else "L"), "geometry"


NORMALISE = PreprocessConfig(spatial_dims=3, percentiles=(1, 99), normalization="minmax")
RESIZE = PreprocessConfig(spatial_dims=3, target_shape=(N_SLICES, IMG, IMG), resize_mode="pad")


def finish_volume(vol):
    """Per-volume 1-99% clip + minmax (nothing fitted on data), 16 evenly spaced
    slices, aspect-preserving resize/pad to 224, uint8."""
    vol = preprocess(vol, config=NORMALISE)["image"]
    idx = np.round(np.linspace(0, len(vol) - 1, N_SLICES)).astype(int)
    vol = preprocess(np.ascontiguousarray(vol[idx]), config=RESIZE)["image"]
    return np.round(np.clip(vol, 0, 1) * 255).astype(np.uint8)


def process_study(study_series, series_root):
    """(3, 16, 224, 224) uint8 in PLANES order, presence mask, log record."""
    picks = select_series(study_series, series_root)
    out = np.zeros((len(PLANES), N_SLICES, IMG, IMG), np.uint8)
    present = np.zeros(len(PLANES), np.float32)
    raw, metas, errors = {}, [], []
    for plane, folder in picks.items():
        try:
            vol, meta = read_series(folder)
            raw[plane] = (vol, meta)
            metas.append(meta)
        except Exception as exc:
            errors.append(f"{plane}: {type(exc).__name__}: {exc}")
    side, side_source = study_side(metas)
    plane_mismatch = 0
    for k, plane in enumerate(PLANES):
        if plane not in raw:
            continue
        vol, meta = raw[plane]
        plane_mismatch += int(meta["geom_plane"] != plane)
        out[k] = finish_volume(to_left_knee(vol, meta["axes"], side))
        present[k] = 1.0
    record = {"side": side or "", "side_source": side_source,
              "planes_present": int(present.sum()), "plane_mismatch": plane_mismatch,
              "error": " | ".join(errors)}
    return out, present, record

In [ ]:
# Data-free checks of the orientation logic (synthetic arrays only).
_v = np.arange(2 * 3 * 4, dtype=np.float32).reshape(2, 3, 4)
# Coronal, standard: rows along +x (L), columns along -z -> flip rows only.
_c, _ax = to_canonical(_v, np.array([1, 0, 0, 0, 0, -1.0]))
assert _ax == (1, 2, 0) and np.array_equal(_c, _v[:, ::-1, :])
# Same coronal acquired mirrored (rows along -x): canonical result must match.
_c2, _ = to_canonical(_v[:, :, ::-1], np.array([-1, 0, 0, 0, 0, -1.0]))
assert np.array_equal(_c2, _c)
# Right knee: mirror along the array axis that carries patient x.
assert np.array_equal(to_left_knee(_c, _ax, "R"), _c[:, :, ::-1])
assert to_left_knee(_c, _ax, "L") is _c
# Sagittal: patient x is the slice axis.
_s, _sax = to_canonical(_v, np.array([0, 1, 0, 0, 0, -1.0]))
assert _sax[0] == 0 and np.array_equal(to_left_knee(_s, _sax, "R"), _s[::-1])
# Side from geometry and tags.
assert study_side([{"tag": "", "centre_x": -80.0}])[0] == "R"
assert study_side([{"tag": "", "centre_x": 5.0}])[0] is None
assert study_side([{"tag": "L", "centre_x": -80.0}]) == ("L", "tag")
_f = finish_volume(np.random.default_rng(0).random((23, 300, 200)).astype(np.float32))
assert _f.shape == (N_SLICES, IMG, IMG) and _f.dtype == np.uint8
print("preprocessing self-test passed")

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
OUT = Path("/kaggle/working")
TARGET_COLUMNS = [
    "ACL", "MCL",
    "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA",
    "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
]
SEED = 42

import time
from concurrent.futures import ThreadPoolExecutor

LIMIT = None  # e.g. 20 for a smoke run
CACHE = OUT / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
series = pd.read_csv(ROOT / "train_series.csv", dtype=str)
for c in ("Fluid_Sensitive", "Fat_Suppression"):
    series[c] = series[c].astype(int)
studies = sorted(series["StudyInstanceUID"].unique())[:LIMIT]
by_study = dict(tuple(series.groupby("StudyInstanceUID")))
print("studies to process:", len(studies), "| version:", PREPROC_VERSION)

In [ ]:
def run_one(study):
    path = CACHE / f"{study}.npz"
    if path.exists():
        return None
    t0 = time.time()
    try:
        vol, present, rec = process_study(by_study[study], ROOT / "train_series")
        tmp = path.with_suffix(".tmp.npz")
        np.savez(tmp, volume=vol, present=present)
        tmp.replace(path)
        rec["status"] = "ok" if present.any() else "empty"
    except Exception as exc:
        rec = {"status": "failed", "error": f"{type(exc).__name__}: {exc}"}
    rec.update(StudyInstanceUID=study, seconds=round(time.time() - t0, 2),
               version=PREPROC_VERSION)
    return rec


manifest_path = OUT / "preprocess_manifest_v1.csv"
records = pd.read_csv(manifest_path).to_dict("records") if manifest_path.exists() else []
t0 = time.time()
with ThreadPoolExecutor(4) as pool:
    for i, rec in enumerate(pool.map(run_one, studies), 1):
        if rec is not None:
            records.append(rec)
        if i % 200 == 0 or i == len(studies):
            pd.DataFrame(records).to_csv(manifest_path, index=False)
            print(f"{i}/{len(studies)} studies, {time.time() - t0:.0f}s")

## Audit the run

In [ ]:
man = pd.read_csv(manifest_path).drop_duplicates("StudyInstanceUID", keep="last")
print("status:", man["status"].value_counts().to_dict())
print("planes present:", man["planes_present"].value_counts().sort_index().to_dict())
print("side source:", man["side_source"].value_counts().to_dict())
print("side:", man["side"].fillna("").replace("", "<unresolved>").value_counts().to_dict())
print("series whose geometry disagrees with the CSV plane:", int(man["plane_mismatch"].sum()))
print("studies with a series error:", int(man["error"].fillna("").ne("").sum()))
assert man["status"].eq("ok").mean() > 0.98, "too many failures: inspect errors first"

In [ ]:
import matplotlib.pyplot as plt

example = np.load(CACHE / f"{man.loc[man['status'].eq('ok'), 'StudyInstanceUID'].iloc[0]}.npz")
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, plane, v in zip(axes, PLANES, example["volume"]):
    ax.imshow(v[N_SLICES // 2], cmap="gray", vmin=0, vmax=255)
    ax.set_title(plane)
    ax.axis("off")
plt.tight_layout()
plt.show()